# Recommendation System Evaluation — Train/Test Split

This notebook evaluates the recommendation system using a **per-node 80/20 train/test split**:

1. **Graph split**: Each node's outgoing edges are shuffled and 20% held out as test (min 1 test edge per node with outgoing edges)
2. **Power-law alpha**: Refitted on the **train graph** degree distribution (no data leakage)
3. **Node2Vec retraining**: Re-trained from scratch on the **train graph** only (SVD-based, pure numpy)
4. **Combined embeddings**: BGE-M3 (unchanged) + degree-normalised train-Node2Vec
5. **Evaluation**: Hits@1, Hits@10, MRR on held-out test edges



In [1]:
import sys, os, random, pickle, json
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

REPO_ROOT    = r'C:\\programming\\github-repos\\graph-ending'
CACHE_DIR    = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki', 'cache', 'eval-1')
os.makedirs(CACHE_DIR, exist_ok=True)

DATA_PATH    = "../../data/data-embeddings.json"
BGE_M3_PATH  = "../../cap-embeddings/BAAI_bge-m3/master_embeddings.parquet"
N2V_PATH     = "../../cap-embeddings/node2vec/master_embeddings.parquet"

utils_parent = os.path.join(REPO_ROOT, 'WikiCS', 'custom-wiki')
sys.path.insert(0, utils_parent)
from utils import (
    load_graph_data,
    load_embeddings,
    build_normalized_embeddings,
    node2vec_svd,
)

print("Setup done.")


Setup done.


## Step 1 — Load Full Graph


In [2]:
nodes, id_to_title, title_to_id, G_full = load_graph_data(DATA_PATH)
print(f"Full graph: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} edges")


Full graph: 11701 nodes, 291039 edges


## Step 2 — Per-Node 80/20 Train/Test Edge Split

For each node `u` with outgoing edges:
- `ceil(0.2 * d_out(u))` edges → **Test**
- Remaining edges → **Train**
- Nodes with `d_out == 0` are excluded from evaluation.



In [3]:
random.seed(42)
np.random.seed(42)

E_train = []
E_test  = []
eval_nodes = set()

for u in G_full.nodes():
    out_edges = list(G_full.out_edges(u))
    if not out_edges:
        continue
    n_out = len(out_edges)
    n_test = max(1, int(np.ceil(0.2 * n_out)))
    random.shuffle(out_edges)
    E_test.extend(out_edges[:n_test])
    E_train.extend(out_edges[n_test:])
    if n_test > 0:
        eval_nodes.add(u)

G_train = nx.DiGraph()
G_train.add_nodes_from(G_full.nodes())
G_train.add_edges_from(E_train)

print(f"E_train: {len(E_train)} edges")
print(f"E_test : {len(E_test)} edges")
print(f"G_train: {G_train.number_of_nodes()} nodes, {G_train.number_of_edges()} edges")
print(f"Eval nodes (>=1 test edge): {len(eval_nodes)}")

SPLIT_FILE = os.path.join(CACHE_DIR, "train_test_split.pkl")
with open(SPLIT_FILE, "wb") as f:
    pickle.dump({'E_train': E_train, 'E_test': E_test,
                  'G_train': G_train, 'eval_nodes': eval_nodes}, f)
print(f"Saved to {SPLIT_FILE}")


E_train: 227933 edges
E_test : 63106 edges
G_train: 11701 nodes, 227933 edges
Eval nodes (>=1 test edge): 11256
Saved to C:\\programming\\github-repos\\graph-ending\WikiCS\custom-wiki\cache\eval-1\train_test_split.pkl


## Step 3 — Power-Law Fit on Train Graph Degrees


In [4]:
import powerlaw

train_degrees = [G_train.degree(n) for n in G_train.nodes()]
fit_train = powerlaw.Fit(np.array(train_degrees, dtype=float), discrete=True, verbose=False)
alpha_train = fit_train.power_law.alpha
xmin_train  = fit_train.power_law.xmin

print(f"Train graph power-law fit:")
print(f"  alpha = {alpha_train:.4f}")
print(f"  xmin  = {xmin_train}")
print(f"  sigma = {fit_train.power_law.sigma:.4f}")

ALPHA_FILE = os.path.join(CACHE_DIR, "alpha_train.pkl")
with open(ALPHA_FILE, "wb") as f:
    pickle.dump({'alpha': alpha_train, 'xmin': xmin_train}, f)
print(f"Saved to {ALPHA_FILE}")


Train graph power-law fit:
  alpha = 2.9763
  xmin  = 90.0
  sigma = 0.0482
Saved to C:\\programming\\github-repos\\graph-ending\WikiCS\custom-wiki\cache\eval-1\alpha_train.pkl


## Step 4 — Load BGE-M3 Embeddings

BGE-M3 is content-only (text-based) — unchanged from the full graph.


In [5]:
df_bge, df_n2v_original = load_embeddings(BGE_M3_PATH, N2V_PATH)
print(f"BGE-M3  : {len(df_bge)} nodes, dim={len(df_bge['embedding'].iloc[0])}")
print(f"Node2Vec (original): {len(df_n2v_original)} nodes, dim={len(df_n2v_original['embedding'].iloc[0])}")


BGE-M3  : 11701 nodes, dim=1024
Node2Vec (original): 11701 nodes, dim=128


## Step 5 — Retrain Node2Vec on Train Graph (SVD-based)

Pure-numpy implementation using alias-sampled random walks + PPMI matrix factorisation via TruncatedSVD.
No C extensions required. Parameters match the original Node2Vec: p=1, q=1, walks_per_node=10.


In [6]:
print("Training Node2Vec on train graph (SVD-based)...")
G_train_undirected = G_train.to_undirected()

n2v_emb = node2vec_svd(
    G_train_undirected,
    num_walks=10,
    walk_length=80,
    embedding_dim=128,
    seed=42,
)
print(f"  Train Node2Vec: {len(n2v_emb)} nodes, dim=128")

n2v_ids = list(n2v_emb.keys())
df_n2v_train = pd.DataFrame({
    'id': n2v_ids,
    'embedding': [n2v_emb[k] for k in n2v_ids]
})

N2V_TRAIN_FILE = os.path.join(CACHE_DIR, "train_node2vec.parquet")
df_n2v_train.to_parquet(N2V_TRAIN_FILE, index=False)
print(f"Saved to {N2V_TRAIN_FILE}")


Training Node2Vec on train graph (SVD-based)...


  Train Node2Vec: 11701 nodes, dim=128


Saved to C:\\programming\\github-repos\\graph-ending\WikiCS\custom-wiki\cache\eval-1\train_node2vec.parquet


## Step 6 — Build Degree-Normalised Combined Embeddings


In [7]:
df_combined = build_normalized_embeddings(df_bge, df_n2v_train, G_train, alpha_train)
print(f"Combined embeddings: {len(df_combined)} nodes, dim={len(df_combined['embedding'].iloc[0])}")

EMB_FILE = os.path.join(CACHE_DIR, "combined_embeddings.pkl")
with open(EMB_FILE, "wb") as f:
    pickle.dump(df_combined, f)
print(f"Saved to {EMB_FILE}")


Combined embeddings: 11701 nodes, dim=1152
Saved to C:\\programming\\github-repos\\graph-ending\WikiCS\custom-wiki\cache\eval-1\combined_embeddings.pkl


## Step 7 — Compute Similarity Matrix


In [8]:
embedding_matrix = np.stack(list(df_combined["embedding"].values))
node_ids_arr = df_combined["id"].values
id_to_idx   = {nid: i for i, nid in enumerate(node_ids_arr)}
idx_to_id   = {i: nid for i, nid in enumerate(node_ids_arr)}

SIM_FILE = os.path.join(CACHE_DIR, "similarity_matrix.pkl")
if os.path.exists(SIM_FILE):
    print("[cache] Loading similarity matrix...")
    with open(SIM_FILE, "rb") as f:
        sim_matrix = pickle.load(f)
    print(f"[cache] Loaded: shape {sim_matrix.shape}")
else:
    print("[comp] Computing similarity matrix...")
    sim_matrix = cosine_similarity(embedding_matrix)
    with open(SIM_FILE, "wb") as f:
        pickle.dump(sim_matrix, f)
    print("[save] Saved.")

np.fill_diagonal(sim_matrix, 0.0)
print(f"Ready: {sim_matrix.shape}")


[cache] Loading similarity matrix...


[cache] Loaded: shape (11701, 11701)
Ready: (11701, 11701)


## Step 8 — Evaluation: Hits@1, Hits@10, MRR

For each **evaluation node** `u` (nodes with at least 1 test edge):
1. Get `u`'s test neighbours from `E_test`
2. Rank all nodes by cosine similarity to `u`
3. Record the rank of each test neighbour

Metrics:
- **Hits@K**: fraction of test neighbours appearing in top-K
- **MRR**: Mean Reciprocal Rank = average of 1/rank across all test neighbours


In [9]:
node_test_neighbors = {}
for u, v in E_test:
    node_test_neighbors.setdefault(u, []).append(v)

eval_nodes_filtered = [n for n in eval_nodes if n in node_test_neighbors]
print(f"Evaluation nodes: {len(eval_nodes_filtered)}")

def compute_ranks(u, test_neighbors):
    if u not in id_to_idx:
        return []
    u_idx = id_to_idx[u]
    sims = sim_matrix[u_idx]
    sorted_indices = np.argsort(sims)[::-1]
    ranks = []
    for v in test_neighbors:
        if v not in id_to_idx:
            continue
        v_idx = id_to_idx[v]
        rank = np.where(sorted_indices == v_idx)[0]
        if len(rank) > 0:
            ranks.append(rank[0] + 1)
    return ranks

print("Computing ranks for all eval nodes...")
hits_at_1  = 0
hits_at_10 = 0
total_test_edges = 0
mrr_sum = 0.0

for i_u, u in enumerate(eval_nodes_filtered):
    test_neighbors = node_test_neighbors[u]
    total_test_edges += len(test_neighbors)
    ranks = compute_ranks(u, test_neighbors)
    for r in ranks:
        if r <= 1:  hits_at_1 += 1
        if r <= 10: hits_at_10 += 1
        mrr_sum += 1.0 / r
    if (i_u + 1) % 2000 == 0:
        print(f"  Processed {i_u+1}/{len(eval_nodes_filtered)}...")

hits_at_1_rate  = hits_at_1  / total_test_edges if total_test_edges > 0 else 0
hits_at_10_rate = hits_at_10 / total_test_edges if total_test_edges > 0 else 0
mrr = mrr_sum / total_test_edges if total_test_edges > 0 else 0

print(f"\n=== EVALUATION RESULTS ===")
print(f"Total test edges  : {total_test_edges}")
print(f"Hits@1  rate     : {hits_at_1_rate:.4f} ({hits_at_1}/{total_test_edges})")
print(f"Hits@10 rate     : {hits_at_10_rate:.4f} ({hits_at_10}/{total_test_edges})")
print(f"MRR              : {mrr:.4f}")


Evaluation nodes: 11256
Computing ranks for all eval nodes...


  Processed 2000/11256...


  Processed 4000/11256...


  Processed 6000/11256...


  Processed 8000/11256...


  Processed 10000/11256...



=== EVALUATION RESULTS ===
Total test edges  : 63106
Hits@1  rate     : 0.0170 (1075/63106)
Hits@10 rate     : 0.1149 (7249/63106)
MRR              : 0.0548


In [10]:
results = {
    'hits_at_1_rate': hits_at_1_rate,
    'hits_at_10_rate': hits_at_10_rate,
    'mrr': mrr,
    'total_test_edges': total_test_edges,
    'n_eval_nodes': len(eval_nodes_filtered),
    'alpha_train': alpha_train,
}

RESULTS_FILE = os.path.join(CACHE_DIR, "eval_results.pkl")
with open(RESULTS_FILE, "wb") as f:
    pickle.dump(results, f)
print(f"Results saved to {RESULTS_FILE}")


Results saved to C:\\programming\\github-repos\\graph-ending\WikiCS\custom-wiki\cache\eval-1\eval_results.pkl


## Step 9 — Qualitative Check

Inspect a sample of evaluation nodes to verify recommendation quality.


In [11]:
def safe_title(node_id):
    return id_to_title.get(node_id, id_to_title.get(str(node_id), str(node_id)))

def get_recs(node_id, top_k=10):
    idx = id_to_idx[node_id]
    sims = sim_matrix[idx]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(idx_to_id[i], round(float(sims[i]), 4)) for i in top_idx]

sample_nodes = eval_nodes_filtered[:5]
for u in sample_nodes:
    title_u = safe_title(u)
    test_neighbors_titles = [safe_title(v) for v in node_test_neighbors[u]]
    recs = get_recs(u, top_k=5)
    recs_titled = [(safe_title(nid), sim) for nid, sim in recs]
    print(f"\nNode: {title_u}")
    print(f"  Test neighbours hidden: {test_neighbors_titles}")
    print(f"  Top-5 recs: {recs_titled}")



Node: Twilio
  Test neighbours hidden: ['Web_service', 'Salesforce.com']
  Top-5 recs: [('ServiceNow', 0.6475), ('SnapLogic', 0.6133), ('Cloudreach', 0.6108), ('Typeform_(service)', 0.6063), ('Workday,_Inc.', 0.6042)]

Node: Program_compatibility_date_range
  Test neighbours hidden: ['Program_Files']
  Top-5 recs: [('Processor_Control_Region', 1.0), ('OpenStego', 0.8616), ('International_Middleware_Conference', 0.8616), ('Kernel_relocation', 0.8446), ('Linux_India', 0.8446)]

Node: SYSTAT_(DEC)
  Test neighbours hidden: ['RSTS/E']
  Top-5 recs: [('Commercial_Operating_System_(COS)', 0.8189), ('TOPS-10', 0.781), ('Asynchronous_System_Trap', 0.7696), ('Virtual_device', 0.7508), ('DEC_BATCH-11/DOS-11', 0.7497)]

Node: Stealth_wallpaper
  Test neighbours hidden: ['Wireless_security']
  Top-5 recs: [('Michael_Schearer', 0.7993), ('Ipsectrace', 0.7773), ('Wi-Fi_deauthentication_attack', 0.7128), ('KARMA_attack', 0.7009), ('Cryptek', 0.6889)]

Node: List_of_column-oriented_DBMSes
  Test neig

In [12]:
print("=== SUMMARY ===")
print(f"Train graph edges : {G_train.number_of_edges()}")
print(f"Test edges        : {len(E_test)}")
print(f"Eval nodes        : {len(eval_nodes_filtered)}")
print(f"Train alpha       : {alpha_train:.4f}")
print(f"Hits@1 rate       : {hits_at_1_rate:.4f}")
print(f"Hits@10 rate      : {hits_at_10_rate:.4f}")
print(f"MRR               : {mrr:.4f}")
n = 11701
random_hit10 = 10 / (n - 1)
print(f"Random Hit@10     : {random_hit10:.6f} ({random_hit10*100:.4f}%)")
print(f"Improvement vs     : {hits_at_10_rate / random_hit10:.0f}x better than random")


=== SUMMARY ===
Train graph edges : 227933
Test edges        : 63106
Eval nodes        : 11256
Train alpha       : 2.9763
Hits@1 rate       : 0.0170
Hits@10 rate      : 0.1149
MRR               : 0.0548
Random Hit@10     : 0.000855 (0.0855%)
Improvement vs     : 134x better than random
